First shot output
![QGIS spur artifacts](Artifacts/qgis_spur_artifacts.png)

Big Picture: I want to make a trip generation visualization tool to understand where are the points where people intersect intersect the most at. I want to ground this analysis so I will be basing them on two locations: Jackson Heights and Bed-Stuy. I will constrain this further by making a 20 (maybe 30) minute perimeter around each neighborhoods primary park (Herbert Von King and Travers). Additionally I will populate the points at which the analysis is being done based off of real census data.
The trip analysis will then be computed in Grasshopper. 

Intent: Find centerlines for NYC including parks in order to construct shortest trip generation
So far there is not a dataset including of centerlines including park paths, which is commonly used for making the shortest path.
This led me to NYC Planimetric Database: Sidewalk. database

Insert visualization

First I'm going to start to see what I have in the list (bc I can be forgetful) as there are some other plugins required for turning a polygon map into centerlines

In [1]:
!conda env list


# conda environments:
#
# * -> active
# + -> frozen
ZCDPGIS              *   /Users/temp/.conda/envs/ZCDPGIS
cdp312                   /Users/temp/.conda/envs/cdp312
                         /Users/temp/Documents/GitHub/cdp-mapping-systems/.conda
base                     /opt/anaconda3
                         /opt/miniconda3/envs/geo_env



In [2]:
!conda list | grep -E "geopandas|shapely|fiona|pyproj|matplotlib"

fiona                            1.10.1           py313h7df67bf_6      conda-forge
geopandas                        1.1.3            pypi_0               pypi
matplotlib-base                  3.10.9           py313h36cb854_0      conda-forge
matplotlib-inline                0.2.1            pypi_0               pypi
pyproj                           3.7.2            py313h6de5794_3      conda-forge
shapely                          2.1.2            py313h72d6987_0


In [3]:
!pip install centerline
!pip show folium
!pip install folium mapclassify

Name: folium
Version: 0.20.0
Summary: Make beautiful maps with Leaflet.js & Python
Home-page: https://github.com/python-visualization/folium
Author: Rob Story
Author-email: wrobstory@gmail.com
License: MIT
Location: /Users/temp/.conda/envs/ZCDPGIS/lib/python3.13/site-packages
Requires: branca, jinja2, numpy, requests, xyzservices
Required-by: cdptools, leafmap


In [4]:
!pip show centerline

Name: centerline
Version: 1.1.1
Summary: Calculate the centerline of a polygon
Home-page: https://github.com/fitodic/centerline
Author: Filip Todic
Author-email: todic.filip@gmail.com
License: MIT License
Location: /Users/temp/.conda/envs/ZCDPGIS/lib/python3.13/site-packages
Requires: Click, Fiona, numpy, scipy, Shapely
Required-by: 


In [5]:
import geopandas as gpd
import pandas as pd

nta = gpd.read_file("/Users/temp/Documents/GitHub/cdp-mapping-systems/Personal Work/Pathmapping_Tool/Data/2020_Neighborhood_Tabulation_Areas_(NTAs)_20260717.geojson")
sidewalk = gpd.read_file("/Users/temp/Documents/GitHub/cdp-mapping-systems/Personal Work/Pathmapping_Tool/Data/NYC_Planimetric_Database__Sidewalk_20260717.geojson")
street_centerline = gpd.read_file("/Users/temp/Documents/GitHub/cdp-mapping-systems/Personal Work/Pathmapping_Tool/Data/Centerline_20260717-2.geojson")

This section is defining all of the datasets to the same coordinate reference system.
To_crs is a command to do that.
"WORKING_CRS" is a variable holding the numered reference system

In [6]:
WORKING_CRS = 2263  # NAD83 State Plane Long Island (feet)

nta = nta.to_crs(WORKING_CRS)
sidewalk = sidewalk.to_crs(WORKING_CRS)
street_centerline = street_centerline.to_crs(WORKING_CRS)

This is filtering the parts of the dataset that I want and putting them into respected categories
Library Functions: {}, for, in, .items(), .str.contains(), gpd.GeoDataFrame(...)
For: "for every time (key, and name_filter) show up
In: do the following funciton


In [ ]:
neighborhoods = {
    "bedstuy": "Bedford-Stuyvesant",
    "jacksonheights": "Jackson Heights"
}

boundaries = {}
for key, name_filter in neighborhoods.items():
    parts = nta[nta["ntaname"].str.contains(name_filter, na=False)] #saying filter the nta, specifically the ntaname column, to only include rows that contain the name_filter string (e.g., "Bedford-Stuyvesant" or "Jackson Heights"). The na=False argument ensures that any rows with NaN values in the ntaname column are excluded from the filter.
    print(f"{key}: {len(parts)} NTA piece(s) found")
    boundaries[key] = gpd.GeoDataFrame(
        geometry=[parts.unary_union],
        crs=nta.crs
    )

bedstuy: 2 NTA piece(s) found
jacksonheights: 1 NTA piece(s) found


/var/folders/jm/2cq5x30j3gx977bqk1tjsk0r0000gn/T/ipykernel_58194/48678306.py:11: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geometry=[parts.unary_union],
/var/folders/jm/2cq5x30j3gx977bqk1tjsk0r0000gn/T/ipykernel_58194/48678306.py:11: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  geometry=[parts.unary_union],


Running into a situation where the street shapefiles have gaps as there are curb cuts. Running this function to prune those gaps

In [8]:
def prune_spurs(lines_gdf, min_length=15, passes=3):
    lines = lines_gdf.explode(index_parts=False).reset_index(drop=True)

    for _ in range(passes):
        endpoint_counts = {}
        for geom in lines.geometry:
            for pt in [geom.coords[0], geom.coords[-1]]:
                key = (round(pt[0], 1), round(pt[1], 1))  # rounded to avoid float mismatch
                endpoint_counts[key] = endpoint_counts.get(key, 0) + 1

        def is_spur(geom):
            start = (round(geom.coords[0][0], 1), round(geom.coords[0][1], 1))
            end = (round(geom.coords[-1][0], 1), round(geom.coords[-1][1], 1))
            dangling = endpoint_counts[start] == 1 or endpoint_counts[end] == 1
            return dangling and geom.length < min_length

        lines = lines[~lines.geometry.apply(is_spur)].reset_index(drop=True)

    return lines

In [10]:
from shapely.geometry import MultiPolygon, MultiLineString, Point
from centerline.geometry import Centerline

def get_endpoints(geom):
    lines = geom.geoms if isinstance(geom, MultiLineString) else [geom]
    points = []
    for line in lines:
        coords = list(line.coords)
        if coords:
            points.append(coords[0])
            points.append(coords[-1])
    return points

def prune_spurs(lines_gdf, min_length=15, passes=3):
    lines = lines_gdf.explode(index_parts=False).reset_index(drop=True)
    for _ in range(passes):
        endpoint_counts = {}
        for geom in lines.geometry:
            for pt in [geom.coords[0], geom.coords[-1]]:
                key = (round(pt[0], 1), round(pt[1], 1))
                endpoint_counts[key] = endpoint_counts.get(key, 0) + 1

        def is_spur(geom):
            start = (round(geom.coords[0][0], 1), round(geom.coords[0][1], 1))
            end = (round(geom.coords[-1][0], 1), round(geom.coords[-1][1], 1))
            dangling = endpoint_counts[start] == 1 or endpoint_counts[end] == 1
            return dangling and geom.length < min_length

        lines = lines[~lines.geometry.apply(is_spur)].reset_index(drop=True)
    return lines

def build_network(boundary_gdf, neighborhood_name):
    clipped = gpd.clip(sidewalk.to_crs(boundary_gdf.crs), boundary_gdf)
    centerline_geoms = []
    for geom in clipped.geometry:
        polys = geom.geoms if isinstance(geom, MultiPolygon) else [geom]
        for poly in polys:
            try:
                centerline_geoms.append(Centerline(poly).geometry)
            except Exception as e:
                pass

    walk_lines = gpd.GeoDataFrame(geometry=centerline_geoms, crs=clipped.crs)
    walk_lines = prune_spurs(walk_lines, min_length=15)
    walk_lines["segment_type"] = "block"

    street_clipped = gpd.clip(street_centerline.to_crs(boundary_gdf.crs), boundary_gdf)
    all_endpoints = [Point(pt) for line in walk_lines.geometry for pt in get_endpoints(line)]
    endpoint_gdf = gpd.GeoDataFrame(geometry=all_endpoints, crs=walk_lines.crs)
    buffer_union = endpoint_gdf.buffer(20).unary_union

    street_bridges = street_clipped[street_clipped.intersects(buffer_union)].copy()
    street_bridges["segment_type"] = "crossing"

    result = gpd.GeoDataFrame(
        pd.concat([
            walk_lines[["geometry", "segment_type"]],
            street_bridges[["geometry", "segment_type"]]
        ], ignore_index=True),
        crs=walk_lines.crs
    )
    result["neighborhood"] = neighborhood_name
    return result

In [11]:
for name, boundary in boundaries.items():
    boundary.to_file(f"data/boundary_{name}.geojson", driver="GeoJSON")

In [12]:
all_networks = [
    build_network(boundary, name)
    for name, boundary in boundaries.items()
]

final_network = pd.concat(all_networks, ignore_index=True).reset_index(drop=True)
final_network["segment_id"] = final_network.index

/var/folders/jm/2cq5x30j3gx977bqk1tjsk0r0000gn/T/ipykernel_58194/1462772762.py:50: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  buffer_union = endpoint_gdf.buffer(20).unary_union
/var/folders/jm/2cq5x30j3gx977bqk1tjsk0r0000gn/T/ipykernel_58194/1462772762.py:50: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  buffer_union = endpoint_gdf.buffer(20).unary_union


In [13]:
print("sidewalk CRS:", sidewalk.crs)
print("bedstuy boundary CRS:", boundaries["bedstuy"].crs)

print("sidewalk bounds:", sidewalk.total_bounds)
print("bedstuy boundary bounds (in sidewalk's CRS):", 
      boundaries["bedstuy"].to_crs(sidewalk.crs).total_bounds)

test_clip = gpd.clip(sidewalk.to_crs(boundaries["bedstuy"].crs), boundaries["bedstuy"])
print("test clip rows:", len(test_clip))

sidewalk CRS: EPSG:2263
bedstuy boundary CRS: EPSG:2263
sidewalk bounds: [ 913328.53693726  120266.48781302 1067488.3873361   272546.45622757]
bedstuy boundary bounds (in sidewalk's CRS): [ 994880.68011462  185942.34479465 1007465.53249862  194575.10505494]
test clip rows: 455


In [14]:
import os
os.makedirs("outputs", exist_ok=True)

final_network.to_file("outputs/walk_network.geojson", driver="GeoJSON")